# Data audit: what the data can and cannot measure

**Purpose.** This notebook records how the assumptions of the project *What happens to professional football players after a serious meniscal injury in Europe's top five leagues?* were checked against the data before any threshold, window or definition was fixed.

**Data (Transfermarkt, via Kaggle).**
- dataset 1, `xfkzujqjvx97n/football-datasets`: `player_injuries`, `player_performances`
- dataset 2, `davidcariboo/player-scores`: `players`, `appearances`, `games`, `transfers`

**Table of contents**

| Step | Question |
|---|---|
| 0 | Which tables do we have, and what do their keys, missing values and date ranges look like? |
| 1 | For meniscal injuries, how many have data before and after the injury? |
| 2 | Do the two datasets agree where they describe the same thing? |
| 3 | From when to when are the data complete? |
| 4 | How good are the meniscal injury records? |
| 5 | How is absence distributed, and where is the severity threshold? |
| 6 | Which competition levels are visible in each table, and how do we label the level after return? |
| 7 | Which role measure and which minimum minutes baseline? |
| 8 | How many players vanish after return, and can a club change be detected? |

Work in progress...

## Step 0: What do we have? Tables, keys, missing values, date ranges

**Question.** Which tables do we have, how many rows, which column types and missing values do they have, are the keys unique where they should be, and which period and competitions do the tables cover?

### 0.1 Load the tables and 0.2 convert dates

In the CSV files dates are text. We convert them to dates and **count how many values failed to convert** (`errors="coerce"` silently turns them into missing values, so without this check the errors would disappear).

In [1]:
# 0.1 Setup and loading 
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

KAGGLE_A = Path("/kaggle/input/datasets/xfkzujqjvx97n/football-datasets")
KAGGLE_B = Path("/kaggle/input/datasets/davidcariboo/player-scores")

def csv_path(name):
    candidates = [KAGGLE_A / name / f"{name}.csv", KAGGLE_B / f"{name}.csv",
                  Path("data") / f"{name}.csv", Path("../data") / f"{name}.csv"]
    found = next((p for p in candidates if p.exists()), None)
    if found is None:
        raise FileNotFoundError(f"{name}.csv not found; looked in: {[str(p) for p in candidates]}")
    return found

player_injuries     = pd.read_csv(csv_path("player_injuries"))
player_performances = pd.read_csv(csv_path("player_performances"))
players             = pd.read_csv(csv_path("players"))
appearances         = pd.read_csv(csv_path("appearances"))
games               = pd.read_csv(csv_path("games"))
transfers           = pd.read_csv(csv_path("transfers"))

tables = {"player_injuries": player_injuries, "player_performances": player_performances,
          "players": players, "appearances": appearances, "games": games, "transfers": transfers}
for name, df in tables.items():
    print(f"{name:20s} rows: {len(df):>9,}   columns: {df.shape[1]}")

# 0.2 Dates: conversion and range 
date_cols = {"player_injuries": ["from_date", "end_date"], "appearances": ["date"], "games": ["date"],
             "transfers": ["transfer_date"], "players": ["date_of_birth"]}
rows = []
for name, cols in date_cols.items():
    df = tables[name]          # the same object as the variable of that name, so the change is visible everywhere
    for c in cols:
        before = df[c].isna().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce")
        rows.append({"table": name, "column": c, "missing before": before,
                     "new missing after conversion": df[c].isna().sum() - before,
                     "earliest": df[c].min(), "latest": df[c].max()})
pd.DataFrame(rows)

player_injuries      rows:   143,195   columns: 7
player_performances  rows: 1,878,719   columns: 20
players              rows:    50,149   columns: 26
appearances          rows: 1,894,350   columns: 13
games                rows:    88,958   columns: 23
transfers            rows:   175,165   columns: 10


,table,column,missing before,new missing after conversion,earliest,latest
0,player_injuries,from_date,22,0,1973-09-02,2025-12-22
1,player_injuries,end_date,1523,0,1973-09-06,2026-09-01
2,appearances,date,0,0,2012-07-03,2026-06-28
3,games,date,0,0,2006-06-09,2026-07-06
4,transfers,transfer_date,0,0,1993-07-01,2030-06-30
5,players,date_of_birth,49,0,1968-07-31,2011-02-23


### 0.3 Columns and 0.4 keys

For every table: column type, share of missing values and number of unique values. Then: are the identifiers unique where they should be, and how many distinct players does each table contain?

In [2]:
# 0.3 Columns: type, missing values, unique values 
for name, df in tables.items():
    print("=" * 90)
    print(name, df.shape)
    display(pd.DataFrame({"type": df.dtypes.astype(str),
                          "missing": df.isna().sum(),
                          "missing %": (df.isna().mean() * 100).round(1),
                          "unique": df.nunique()}))

# 0.4 Keys and players per table 
print("players:      duplicate player_id     =", players["player_id"].duplicated().sum())
print("games:        duplicate game_id       =", games["game_id"].duplicated().sum())
print("appearances:  duplicate appearance_id =", appearances["appearance_id"].duplicated().sum())
print("transfers:    duplicate (player_id, transfer_date, from_club_id, to_club_id) =",
      transfers.duplicated(["player_id", "transfer_date", "from_club_id", "to_club_id"]).sum())
print("player_performances: duplicate (player_id, season_name, competition_id, team_id) =",
      player_performances.duplicated(["player_id", "season_name", "competition_id", "team_id"]).sum())
print("player_injuries: fully identical rows        =", player_injuries.duplicated().sum())

pd.Series({n: tables[n]["player_id"].nunique() for n in
           ["player_injuries", "player_performances", "players", "appearances", "transfers"]}, name="unique players")

player_injuries (143195, 7)


,type,missing,missing %,unique
player_id,int64,0,0.0,34561
season_name,str,0,0.0,46
injury_reason,str,0,0.0,349
from_date,datetime64[us],22,0.0,8133
end_date,datetime64[us],1523,1.1,8114
days_missed,float64,22,0.0,866
games_missed,int64,0,0.0,135


player_performances (1878719, 20)


,type,missing,missing %,unique
player_id,int64,0,0.0,88375
season_name,str,0,0.0,102
competition_id,str,0,0.0,1643
competition_name,str,0,0.0,1988
team_id,int64,0,0.0,13511
team_name,str,0,0.0,15411
nb_in_group,int64,0,0.0,51
nb_on_pitch,int64,0,0.0,52
goals,float64,138202,7.4,48
assists,int64,0,0.0,29


players (50149, 26)


...

,type,missing,missing %,unique
game_id,int64,0,0.0,88958
competition_id,str,0,0.0,70
season,int64,0,0.0,18
round,str,0,0.0,128
date,datetime64[us],0,0.0,4445
home_club_id,int64,0,0.0,2980
away_club_id,int64,0,0.0,2634
home_club_goals,int64,0,0.0,16
away_club_goals,int64,0,0.0,18
home_club_position,float64,25377,28.5,21


transfers (175165, 10)


,type,missing,missing %,unique
player_id,int64,0,0.0,23379
transfer_date,datetime64[us],0,0.0,5634
transfer_season,str,0,0.0,35
from_club_id,int64,0,0.0,17408
to_club_id,int64,0,0.0,13208
from_club_name,str,0,0.0,18285
to_club_name,str,0,0.0,14086
transfer_fee,float64,61526,35.1,1503
market_value_in_eur,float64,68160,38.9,191
player_name,str,0,0.0,23036


players:      duplicate player_id     = 0
games:        duplicate game_id       = 0
appearances:  duplicate appearance_id = 0
transfers:    duplicate (player_id, transfer_date, from_club_id, to_club_id) = 0
player_performances: duplicate (player_id, season_name, competition_id, team_id) = 0
player_injuries: fully identical rows        = 111


player_injuries        34561
player_performances    88375
players                50149
appearances            29531
transfers              23379
Name: unique players, dtype: int64

### 0.5 Competition coverage in `appearances`

`games` lists matches of many competitions, but `appearances` (one row per player per match) may cover only some of them. We check which competitions never appear in `appearances` and  what share of matches has no `appearances` rows at all, for the top-5 leagues separately from the rest, within the period that `appearances` covers.

In [3]:
# 0.5 Competition coverage in `appearances` -----------------------------------
TOP5 = ["GB1", "ES1", "IT1", "L1", "FR1"]     # Premier League, La Liga, Serie A, Bundesliga, Ligue 1

comp_type = games.drop_duplicates("competition_id").set_index("competition_id")["competition_type"]
in_app = set(appearances["competition_id"])
never = sorted(set(comp_type.index) - in_app)
print("competitions in games:", len(comp_type), "| in appearances:", len(in_app), "| in games but never in appearances:", len(never))
print("never in appearances:", never)

g = games[games["date"] >= appearances["date"].min()].copy()      # only the period that appearances covers
g["has_rows"] = g["game_id"].isin(set(appearances["game_id"]))
g["group"] = np.where(g["competition_id"].isin(TOP5), "top-5 leagues",
                      np.where(g["competition_type"] == "domestic_league", "other domestic leagues", g["competition_type"].astype(str)))
cov = g.groupby("group")["has_rows"].agg(games="size", with_rows="sum")
cov["% games without any row"] = (100 * (1 - cov["with_rows"] / cov["games"])).round(1)
display(cov)

competitions in games: 70 | in appearances: 48 | in games but never in appearances: 22
never in appearances: ['A1', 'AFAC', 'ARG1', 'AUS1', 'BRA1', 'C1', 'COL1', 'COPA', 'EURO', 'JAP1', 'KR1', 'MEX1', 'MLS1', 'NO1', 'PL1', 'RO1', 'RSK1', 'SA1', 'SE1', 'SER1', 'TS1', 'UCOL']


,games,with_rows,% games without any row
group,,,
domestic_cup,12904,7870,39.0
international_cup,3117,2872,7.9
national_team_competition,552,124,77.5
other,7599,5671,25.4
other domestic leagues,38141,30810,19.2
top-5 leagues,25241,25239,0.0


### 0.6 Two crucial details of `player_performances`
We check two details of `player_performances` that matter later: how seasons are written, and how often `minutes_played` is missing.

In [4]:
# 0.6 Two details of player_performances --------------------------------------
pp = player_performances
season_format = np.select([pp["season_name"].str.match(r"^\d{2}/\d{2}$"), pp["season_name"].str.match(r"^\d{4}$")],
                          ["two-year, e.g. 19/20", "single year, e.g. 2019"], default="other")
display(pd.Series(season_format).value_counts().rename("rows").to_frame())
print(f"minutes_played missing in player_performances: {pp['minutes_played'].isna().mean():.1%} of rows")

,rows
"two-year, e.g. 19/20",1546852
"single year, e.g. 2019",331867


minutes_played missing in player_performances: 62.3% of rows


### Findings

- Six tables; the largest are `player_performances` and `appearances` (about 1.9 million rows each). All date columns converted without losing a single value; `end_date` is missing in 1.1% of injury rows (`from_date` in 22 rows).
- Keys are unique (`player_id`, `game_id`, `appearance_id` and the combined keys in `transfers` and `player_performances`), with one exception: `player_injuries` has 111 rows identical in every column to another row.
- `appearances` covers 48 of the 70 competitions in `games` and is almost complete only for the top-5 leagues (2 of 25,241 matches missing). Other domestic leagues miss 19.2% of matches, domestic cups 39.0% and national teams 77.5%.
- In `player_performances` seasons are written in two ways ("19/20" and "2019"), and `minutes_played` is missing in 62.3% of rows.

## Step 1: Who has data before and after the injury?

**Question.** For meniscal injuries: how many players appear in each table, and for how many injuries do we have data in the window *before* and the window *after* the injury?


### 1.1 Which rows are meniscal injuries?

In [5]:
# every injury label that contains "menisc" (any capitalisation)
mask_menisc = player_injuries["injury_reason"].str.contains("menisc", case=False, na=False)
player_injuries.loc[mask_menisc, "injury_reason"].value_counts()

injury_reason
Meniscus injury                 942
Meniscus tear                   451
Meniscus damage                 293
Tear of the lateral meniscus    144
Meniscus irritation              33
Name: count, dtype: int64

In [6]:
MENISCUS_LABELS = ["Meniscus damage", "Meniscus tear", "Meniscus injury",
                   "Tear of the lateral meniscus", "Meniscus irritation"]

meniscus = player_injuries[player_injuries["injury_reason"].isin(MENISCUS_LABELS)]

print("meniscal injuries (rows):", len(meniscus))
print("players:", meniscus["player_id"].nunique())


meniscal injuries (rows): 1863
players: 1657


In [7]:
print("does the label list catch everything that contains 'menisc'?", len(meniscus) == mask_menisc.sum())

does the label list catch everything that contains 'menisc'? True


### 1.2 In which tables do these players appear at all?

Here we only ask whether a `player_id` exists in a table (ever, with no time windows).

In [8]:
ids = meniscus["player_id"].unique()

pd.Series({
    "players with a meniscal injury": len(ids),
    "...in players":                  np.isin(ids, players["player_id"]).sum(),
    "...in appearances":              np.isin(ids, appearances["player_id"]).sum(),
    "...in player_performances":      np.isin(ids, player_performances["player_id"]).sum(),
}, name="number of players")

players with a meniscal injury    1657
...in players                     1027
...in appearances                  845
...in player_performances         1657
Name: number of players, dtype: int64

### 1.3 Clean the injuries and build the windows

We drop injuries without a start or end date (no windows can be built for them) and full duplicates, and count how many that was. Every injury gets an `inj_id` so that results can be joined later.

In [9]:
KEY = ["player_id", "from_date", "end_date", "injury_reason"]

no_dates = meniscus[["from_date", "end_date"]].isna().any(axis=1).sum()
with_dates = meniscus.dropna(subset=["from_date", "end_date"])
duplicates = with_dates.duplicated(KEY).sum()
print("injuries without a start or end date:", no_dates)
print("full duplicates:", duplicates)

m = with_dates.drop_duplicates(KEY).reset_index(drop=True)
m["inj_id"] = m.index
print("injuries kept for the analysis:", len(m))

injuries without a start or end date: 25
full duplicates: 0
injuries kept for the analysis: 1838


In [10]:
PRE_MONTHS = 12
POST_MONTHS = 24

m["pre_start"] = m["from_date"] - pd.DateOffset(months=PRE_MONTHS)
m["post_end"] = m["end_date"] + pd.DateOffset(months=POST_MONTHS)

m[["player_id", "from_date", "end_date", "pre_start", "post_end"]].head()

,player_id,from_date,end_date,pre_start,post_end
0,100127,2012-01-10,2012-03-10,2011-01-10,2014-03-10
1,10076,2014-10-17,2015-02-25,2013-10-17,2017-02-25
2,100988,2022-10-05,2023-08-06,2021-10-05,2025-08-06
3,101253,2018-03-18,2018-09-18,2017-03-18,2020-09-18
4,10197,1998-11-25,1999-01-03,1997-11-25,2001-01-03


### 1.4 `appearances`: does the player have a match in the window before and in the window after?

For every injury we take all matches of that player (`merge` on `player_id`) and keep those that fall into the window. The injury gets `True` if at least one match remains.

In [11]:
a = appearances.loc[appearances["player_id"].isin(m["player_id"]), ["player_id", "date"]]

x = m[["inj_id", "player_id", "pre_start", "from_date", "end_date", "post_end"]].merge(a, on="player_id")

in_pre = (x["date"] >= x["pre_start"]) & (x["date"] < x["from_date"])
in_post = (x["date"] > x["end_date"]) & (x["date"] <= x["post_end"])

m["app_pre"] = m["inj_id"].isin(x.loc[in_pre, "inj_id"])
m["app_post"] = m["inj_id"].isin(x.loc[in_post, "inj_id"])

print("injuries:", len(m))
pd.DataFrame({"n": [m["app_pre"].sum(), m["app_post"].sum()],
              "%": [round(100 * m["app_pre"].mean(), 1), round(100 * m["app_post"].mean(), 1)]},
             index=["appearances: match BEFORE", "appearances: match AFTER"])

injuries: 1838


,n,%
appearances: match BEFORE,470,25.6
appearances: match AFTER,565,30.7


### 1.5 `player_performances`: seasons

In this table a season is written in two ways: `"19/20"` and `"2019"`. We turn both into **the year in which the season starts** (`"19/20"` → 2019, `"2019"` → 2019). For dates, the European season runs from July to June, so a date in month ≥ 7 belongs to the season numbered by its year, and an earlier date to the season numbered by year − 1.

In [12]:
def season_start(name):
    name = str(name)
    if len(name) == 4:                      # written as "2019"
        return int(name)
    yy = int(name[:2])                      # written as "19/20"
    return 1900 + yy if yy >= 50 else 2000 + yy

season_map = {s: season_start(s) for s in player_performances["season_name"].unique()}
player_performances["season_start"] = player_performances["season_name"].map(season_map)

def european_season(dates):
    return np.where(dates.dt.month >= 7, dates.dt.year, dates.dt.year - 1)

m["inj_season"] = european_season(m["from_date"])      # season in which the injury happened
m["ret_season"] = european_season(m["end_date"])       # season in which the player returned

# quick check: a few season names and their numbers
pd.Series(season_map).sort_values().iloc[[0, 1, 2, 50, 70, -3, -2, -1]]

1960     1960
60/61    1960
61/62    1961
2000     2000
10/11    2010
2025     2025
25/26    2025
2026     2026
dtype: int64

In [13]:
p = player_performances.loc[player_performances["player_id"].isin(m["player_id"]),
                            ["player_id", "season_start", "minutes_played"]]

y = m[["inj_id", "player_id", "inj_season", "ret_season"]].merge(p, on="player_id")

pre_season = y["season_start"] == y["inj_season"] - 1
post_seasons = (y["season_start"] > y["ret_season"]) & (y["season_start"] <= y["ret_season"] + 2)
has_minutes = y["minutes_played"].notna()

m["perf_pre"] = m["inj_id"].isin(y.loc[pre_season, "inj_id"])
m["perf_post"] = m["inj_id"].isin(y.loc[post_seasons, "inj_id"])
m["perf_pre_min"] = m["inj_id"].isin(y.loc[pre_season & has_minutes, "inj_id"])
m["perf_post_min"] = m["inj_id"].isin(y.loc[post_seasons & has_minutes, "inj_id"])

flags = {"performances: season row BEFORE":       "perf_pre",
         "performances: season row AFTER":        "perf_post",
         "performances with minutes: BEFORE":     "perf_pre_min",
         "performances with minutes: AFTER":      "perf_post_min"}
pd.DataFrame({"n": [m[c].sum() for c in flags.values()],
              "%": [round(100 * m[c].mean(), 1) for c in flags.values()]}, index=list(flags.keys()))

,n,%
performances: season row BEFORE,1755,95.5
performances: season row AFTER,1679,91.3
performances with minutes: BEFORE,1277,69.5
performances with minutes: AFTER,1324,72.0


### 1.6 Coverage table

The windows extend beyond the date range of `appearances` (2012-07 to 2026-06), so for some injuries data **cannot** exist for reasons of dates alone. The coverage table therefore uses only injuries whose two windows fit inside the `appearances` period . We also count how many injuries would fit if only 12 months after the return were required .

In [14]:
app_min, app_max = appearances["date"].min(), appearances["date"].max()
m["in_range"] = (m["pre_start"] >= app_min) & (m["post_end"] <= app_max)
m["in_range_12"] = (m["pre_start"] >= app_min) & (m["end_date"] + pd.DateOffset(months=12) <= app_max)

print("meniscal injuries after cleaning:", len(m))
print("...with both windows inside the appearances period (12 before, 24 after):", m["in_range"].sum())
print("...if only 12 months after were required (12 before, 12 after):", m["in_range_12"].sum())

def coverage(df):
    checks = {
        "appearances: match BEFORE":                  df["app_pre"],
        "appearances: match AFTER":                   df["app_post"],
        "appearances: BEFORE and AFTER":              df["app_pre"] & df["app_post"],
        "performances: season row BEFORE":            df["perf_pre"],
        "performances: season row AFTER":             df["perf_post"],
        "performances: BEFORE and AFTER":             df["perf_pre"] & df["perf_post"],
        "performances with minutes: BEFORE":          df["perf_pre_min"],
        "performances with minutes: AFTER":           df["perf_post_min"],
        "performances with minutes: BEFORE and AFTER": df["perf_pre_min"] & df["perf_post_min"],
    }
    return pd.DataFrame({"n": [v.sum() for v in checks.values()],
                         "%": [round(100 * v.mean(), 1) for v in checks.values()]},
                        index=list(checks.keys()))

r = m[m["in_range"]]
coverage(r)

meniscal injuries after cleaning: 1838
...with both windows inside the appearances period (12 before, 24 after): 1277
...if only 12 months after were required (12 before, 12 after): 1423


,n,%
appearances: match BEFORE,395,30.9
appearances: match AFTER,430,33.7
appearances: BEFORE and AFTER,320,25.1
performances: season row BEFORE,1214,95.1
performances: season row AFTER,1196,93.7
performances: BEFORE and AFTER,1142,89.4
performances with minutes: BEFORE,876,68.6
performances with minutes: AFTER,975,76.4
performances with minutes: BEFORE and AFTER,718,56.2


### 1.7 The same count for players of the top-5 leagues only


The table above covers **all** meniscal injuries in dataset 1, including players of lower leagues and from outside Europe. Our question is about the top 5, so we need a denominator: injuries of players who played in a match of one of the five leagues in the 12 months before the injury (`top5_pre`: at least `MIN_TOP5_MINUTES` minutes in `appearances`). The level is judged **only from the period before the injury**, so that players are not selected by what happens afterwards.

League ids in Transfermarkt: `GB1` (Premier League), `ES1` (LaLiga), `IT1` (Serie A), `L1` (Bundesliga), `FR1` (Ligue 1).

In [15]:
# check that these ids are really those leagues
player_performances.loc[player_performances["competition_id"].isin(TOP5)].groupby("competition_id")["competition_name"].first()

competition_id
ES1            LaLiga
FR1           Ligue 1
GB1    Premier League
IT1           Serie A
L1         Bundesliga
Name: competition_name, dtype: str

In [16]:

MIN_TOP5_MINUTES = 1          # at least this many minutes in top-5 league matches in the window before the injury

a5 = appearances.loc[appearances["player_id"].isin(m["player_id"]) & appearances["competition_id"].isin(TOP5),
                     ["player_id", "date", "minutes_played"]]
x5 = m[["inj_id", "player_id", "pre_start", "from_date"]].merge(a5, on="player_id")
x5 = x5[(x5["date"] >= x5["pre_start"]) & (x5["date"] < x5["from_date"])]
m["min_top5_pre"] = m["inj_id"].map(x5.groupby("inj_id")["minutes_played"].sum()).fillna(0)
m["top5_pre"] = m["min_top5_pre"] >= MIN_TOP5_MINUTES

r = m[m["in_range"]]
t5 = r[r["top5_pre"]]
print(f"injuries with windows in range: {len(r)}, of which the player played in a top-5 league in the 12 months before: {len(t5)}")

pd.concat({"all in range": coverage(r), "top 5 in the 12 months before": coverage(t5)}, axis=1)

injuries with windows in range: 1277, of which the player played in a top-5 league in the 12 months before: 197


all in range       top 5 in the 12 months before       
                                                       n     %                             n      %
appearances: match BEFORE                            395  30.9                           197  100.0
appearances: match AFTER                             430  33.7                           160   81.2
appearances: BEFORE and AFTER                        320  25.1                           160   81.2
performances: season row BEFORE                     1214  95.1                           196   99.5
performances: season row AFTER                      1196  93.7                           183   92.9
performances: BEFORE and AFTER                      1142  89.4                           182   92.4
performances with minutes: BEFORE                    876  68.6                           144   73.1
performances with minutes: AFTER                     975  76.4                           153   77.7
performances with minutes: BEFORE and AFTER          718  56.2                           127   64.5

Two remarks on this group: (1) "appearances: match BEFORE" is 100% here **by definition** (that is how we chose the group), so it says nothing about the data; (2) a player who did not play a top-5 match in the 12 months before the injury (for example a fresh signing, or a player who was out for a long time earlier) is not in this group.

### 1.8 Where are the players who have no match in `appearances` after the injury?

The group: injuries of top-5 players (`top5_pre`) with **no** match in `appearances` in the 24 months after the return. We check whether we can find them in `player_performances` (rows of the two seasons after the return season) and in `transfers`.

In [17]:
lost = m[m["in_range"] & m["top5_pre"] & ~m["app_post"]].copy()
print("injuries in this group:", len(lost), "| players:", lost["player_id"].nunique())

injuries in this group: 37 | players: 35


In [18]:
# player_performances rows from the two seasons after the return season
pp = player_performances.loc[player_performances["player_id"].isin(lost["player_id"]),
                             ["player_id", "season_start", "competition_id", "nb_on_pitch"]]
z = lost[["inj_id", "player_id", "ret_season"]].merge(pp, on="player_id")
z = z[(z["season_start"] > z["ret_season"]) & (z["season_start"] <= z["ret_season"] + 2)].copy()

z["is_top5_row"] = z["competition_id"].isin(TOP5)
z["top5_matches"] = np.where(z["is_top5_row"], z["nb_on_pitch"], 0)

agg = z.groupby("inj_id").agg(rows=("competition_id", "size"),
                              top5_rows=("is_top5_row", "sum"),
                              top5_matches=("top5_matches", "sum"))

lost = lost.merge(agg, left_on="inj_id", right_index=True, how="left")
lost[["rows", "top5_rows", "top5_matches"]] = lost[["rows", "top5_rows", "top5_matches"]].fillna(0)

lost["status_after"] = np.select(
    [lost["rows"] == 0, lost["top5_matches"] > 0, lost["top5_rows"] > 0],
    ["no row in performances", "plays in top 5 (missing from appearances)", "in a top-5 squad, 0 appearances"],
    default="plays outside the top 5")

lost["status_after"].value_counts()

status_after
plays outside the top 5                      24
no row in performances                        8
in a top-5 squad, 0 appearances               4
plays in top 5 (missing from appearances)     1
Name: count, dtype: int64

In [19]:
# those we cannot see in performances: in which season does their trace in that table end?
last_perf = player_performances.groupby("player_id")["season_start"].max()

gone = lost[lost["status_after"] == "no row in performances"]
gone.assign(last_season_in_performances=gone["player_id"].map(last_perf))[
    ["player_id", "from_date", "end_date", "last_season_in_performances"]]

,player_id,from_date,end_date,last_season_in_performances
474,22550,2016-09-15,2017-02-01,2016
479,22598,2019-02-10,2019-03-26,2018
830,34787,2023-04-29,2023-06-30,2022
1209,51426,2019-10-02,2020-01-09,2019
1236,524986,2022-01-24,2022-02-27,2021
1237,524986,2021-10-03,2021-11-03,2021
1250,5308,2018-02-19,2018-03-17,2017
1524,67089,2022-11-10,2023-06-10,2022


In [20]:
# transfers: do these injuries have a transfer between the injury start and the end of the window after?
t = transfers.loc[transfers["player_id"].isin(lost["player_id"]), ["player_id", "transfer_date", "to_club_name"]]
x2 = lost[["inj_id", "player_id", "from_date", "post_end"]].merge(t, on="player_id")
x2 = x2[(x2["transfer_date"] > x2["from_date"]) & (x2["transfer_date"] <= x2["post_end"])]

print("players of this group present in transfers at all:", lost["player_id"].drop_duplicates().isin(transfers["player_id"]).sum(),
      "of", lost["player_id"].nunique())
print("injuries with at least one transfer in the window:", x2["inj_id"].nunique(), "of", len(lost),
      "| of these transfers, to 'Retired' or 'Without Club':", int(x2["to_club_name"].isin(["Retired", "Without Club"]).sum()))

# does transfers mark the end of a career at all?
transfers.loc[transfers["to_club_name"].isin(["Retired", "Without Club", "Career break", "Unknown"]), "to_club_name"].value_counts()

players of this group present in transfers at all: 8 of 35
injuries with at least one transfer in the window: 5 of 37 | of these transfers, to 'Retired' or 'Without Club': 0


to_club_name
Without Club    3785
Unknown          268
Retired          184
Career break      63
Name: count, dtype: int64

### Findings

- **Meniscal injuries.** Five labels give 1,863 rows (1,657 players). 25 rows lack a start or end date and there are no full duplicates, so 1,838 injuries remain. Of the 1,657 players, 1,027 appear in `players`, 845 in `appearances` and all in `player_performances`.
- **Windows.** Both windows (12 months before, 24 after) lie inside the `appearances` period for 1,277 injuries (1,423 if only 12 months after were required). Only these are used below.
- **Coverage, all 1,277.** `appearances` has a match before the injury for 30.9%, after for 33.7% and both for 25.1%. `player_performances` has a season row before and after for 89.4% (56.2% with non-missing `minutes_played`).
- **Top-5 players** (at least 1 minute in a top-5 league match in the 12 months before): 197 of the 1,277 injuries. For them the data are much better: `appearances` has a match after the injury for 160 (81.2%), and `player_performances` has season rows before and after for 182 (92.4%).
- **The 37 top-5 injuries (35 players) without a match after.** According to `player_performances` 24 play outside the top 5, 4 are in a top-5 squad without an appearance, 1 plays in the top 5 (missing from `appearances`) and 8 have no row in the two seasons after. `transfers` does not explain them: 8 of the 35 players appear in it, 5 injuries have a transfer in the window and none goes to "Retired" or "Without Club".

## Step 2: Do `appearances` and `player_performances` agree?

**Question.** Both tables describe the same fact: how many times a player played in a given competition and season, and how many minutes. Do they say the same thing?


**How we compare.**
- **Key:** `(player_id, season, competition)`. `appearances` has one row per match, so we sum it to this level.
- **Top-5 leagues only.** Only there both tables should be complete (`appearances` covers 48 competitions, `player_performances` more than 1,600).
- **Seasons.** Which seasons can be compared is decided in 2.1.
- **Two measures:** the number of appearances (`nb_on_pitch` against the number of `appearances` rows) and minutes.

### 2.1 Which seasons can we compare, and how do we assign a match to a season?

**Season of a match.** We take it from `games.season`, not from the date. The July-to-June rule would put the matches of 2019/20 that were finished in July and August 2020 (the season was extended by the pandemic) into season 2020/21. The first output below counts how many appearance rows that concerns.

**Which seasons.** For the top-5 leagues we count appearances per season in both tables and divide by the number of matches in `games`. One match has at most 22 starters plus the substitutes used, so this number cannot grow without limit. A table that is complete should give a similar number in every season.

In [21]:
# 2.1 Season of a match: from games.season, not from the date ------------------
season_by_game = games.set_index("game_id")["season"]
appearances["season_start"] = appearances["game_id"].map(season_by_game)
print("matches in appearances without a season:", appearances["season_start"].isna().sum())

by_date = np.where(appearances["date"].dt.month >= 7, appearances["date"].dt.year, appearances["date"].dt.year - 1)
differs = (by_date != appearances["season_start"]) & appearances["competition_id"].isin(TOP5)
print("top-5 rows where the July-June rule gives another season than games.season:", differs.sum())
display(appearances.loc[differs].groupby("season_start")["date"].agg(["size", "min", "max"]))

# 2.1 Which seasons can we compare? Appearances per match in the top-5 leagues -----
chk = pd.DataFrame({
    "matches": games[games["competition_id"].isin(TOP5)].groupby("season").size(),
    "app_rows": appearances[appearances["competition_id"].isin(TOP5)].groupby("season_start").size(),
    "perf_apps": player_performances[player_performances["competition_id"].isin(TOP5)].groupby("season_start")["nb_on_pitch"].sum(),
}).loc[2012:2025]
chk["app_per_match"] = (chk["app_rows"] / chk["matches"]).round(1)
chk["perf_per_match"] = (chk["perf_apps"] / chk["matches"]).round(1)
chk

matches in appearances without a season: 0
top-5 rows where the July-June rule gives another season than games.season: 6726


,size,min,max
season_start,,,
2019,6726,2020-07-01,2020-08-02


,matches,app_rows,perf_apps,app_per_match,perf_per_match
2012,1826.0,50353.0,43823,27.6,24.0
2013,1826.0,50558.0,44537,27.7,24.4
2014,1826.0,48261.0,45966,26.4,25.2
2015,1826.0,50624.0,46940,27.7,25.7
2016,1826.0,50645.0,47624,27.7,26.1
2017,1826.0,50611.0,48520,27.7,26.6
2018,1826.0,50573.0,48847,27.7,26.8
2019,1725.0,48915.0,47982,28.4,27.8
2020,1826.0,53826.0,53412,29.5,29.3
2021,1826.0,53754.0,53840,29.4,29.5


`appearances` has 26 to 31 rows per match in every season from 2012 to 2025. `player_performances` has fewer in the early seasons (24.0 in 2012, about 30 from 2020) and **almost nothing for season 2025/26**: 3,864 appearances against 54,302 in `appearances` (2.2 per match). Season 2025/26 is therefore missing from `player_performances`, and the comparison ends with 2024/25 (`LAST_SEASON = 2024` below).

### 2.2 Build comparable groups: player × season × league

We bring both tables to the same level: one row per (player, season, league). `appearances` has one row per match, so we count the rows (`app_games`) and sum the minutes (`app_minutes`). In `player_performances` a player can have several rows in one season and league (when he played for several clubs), so we sum them: the number of rows (`perf_rows`), the appearances (`perf_games`, the sum of `nb_on_pitch`), the minutes (`perf_minutes`) and the number of rows with known minutes (`perf_minutes_known`). The comparison covers seasons 2012-2024.

In [22]:
# 2.2 Groups (player, season, league) in both tables -----------------------------
CMP_KEY = ["player_id", "season_start", "competition_id"]
FIRST_SEASON, LAST_SEASON = 2012, 2024

# appearances: one row per match -> one row per (player, season, competition)
a_top5 = appearances[appearances["competition_id"].isin(TOP5)]
app_g = a_top5.groupby(CMP_KEY).agg(app_games=("appearance_id", "size"),
                                    app_minutes=("minutes_played", "sum")).reset_index()

# player_performances: the same level; a player can have several rows in one season and league (several clubs)
p_top5 = player_performances[player_performances["competition_id"].isin(TOP5)
                             & player_performances["season_start"].between(FIRST_SEASON, LAST_SEASON)]
perf_g = p_top5.groupby(CMP_KEY).agg(perf_rows=("nb_on_pitch", "size"),
                                     perf_games=("nb_on_pitch", "sum"),
                                     perf_minutes=("minutes_played", "sum"),
                                     perf_minutes_known=("minutes_played", "count")).reset_index()

print("groups (player, season, league) in appearances:", len(app_g))
print("groups in player_performances (top 5, 2012-2024):", len(perf_g))

groups (player, season, league) in appearances: 37419
groups in player_performances (top 5, 2012-2024): 38866


### 2.3 Do the same groups exist in both tables?

We join the two tables on the key. `left_only` = the group is only in `player_performances`, `right_only` = only in `appearances`, `both` = in both. A group that is only in `player_performances` may simply be a squad member who never played (`nb_on_pitch` = 0), so we split `left_only` by that. The join is restricted to seasons 2012-2024.

In [23]:

c = perf_g.merge(app_g, on=CMP_KEY, how="outer", indicator=True)
c = c[c["season_start"].between(FIRST_SEASON, LAST_SEASON)]
display(c["_merge"].value_counts().to_frame("groups"))

lo = c[c["_merge"] == "left_only"]
print("left_only groups:", len(lo), "| of which nb_on_pitch = 0 (squad member, never played):", int((lo["perf_games"] == 0).sum()),
      f"({(lo['perf_games'] == 0).mean():.1%})", "| played but missing from appearances:", int((lo["perf_games"] > 0).sum()))
pf_played = c[(c["_merge"] != "right_only") & (c["perf_games"] > 0)]
print(f"after removing never-played squad rows, groups of player_performances that have a counterpart in appearances: "
      f"{(pf_played['_merge'] == 'both').mean():.1%}")
c[c["_merge"] == "right_only"].groupby("season_start").size().rename("groups only in appearances").to_frame().T

,groups
_merge,
both,32509
left_only,6357
right_only,2144


left_only groups: 6357 | of which nb_on_pitch = 0 (squad member, never played): 5900 (92.8%) | played but missing from appearances: 457
after removing never-played squad rows, groups of player_performances that have a counterpart in appearances: 98.6%


season_start,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
groups only in appearances,411,364,269,233,193,140,117,77,61,52,53,89,85


### 2.4 Number of appearances: is `nb_on_pitch` equal to the number of rows in `appearances`?

We take the groups that are in both tables and subtract: `diff_games` = rows in `appearances` minus `nb_on_pitch` in `player_performances`. A negative value means `appearances` has fewer matches than `player_performances`.

In [24]:
both = c[c["_merge"] == "both"].copy()
both["diff_games"] = both["app_games"] - both["perf_games"]

print("groups in both tables:", len(both))
print("same number of appearances:", round((both["diff_games"] == 0).mean() * 100, 1), "%")
print("groups differing by 3 or more matches:", int((both["diff_games"].abs() >= 3).sum()))

both["diff_games"].clip(-3, 3).value_counts().sort_index().rename("groups").to_frame()

groups in both tables: 32509
same number of appearances: 99.8 %
groups differing by 3 or more matches: 5


,groups
diff_games,
-3.0,5
-2.0,10
-1.0,32
0.0,32453
1.0,9


### 2.5 Minutes: sum from `appearances` against `minutes_played` from `player_performances`

We take only groups in which `player_performances` has minutes in every row (not missing) and their sum is above zero, and compute the ratio `app_minutes / perf_minutes`. If the tables agreed, the ratio would be close to 1.

In [25]:
mins = both[(both["perf_minutes_known"] == both["perf_rows"]) & (both["perf_minutes"] > 0)].copy()
mins["ratio"] = mins["app_minutes"] / mins["perf_minutes"]

print("groups with minutes in both tables:", len(mins), "of", len(both))
mins["ratio"].round(1).value_counts(normalize=True).head(10).round(3).rename("share of groups").to_frame()

groups with minutes in both tables: 18082 of 32509


,share of groups
ratio,
1.0,0.441
2.0,0.163
3.0,0.100
4.0,0.066
5.0,0.049
6.0,0.036
7.0,0.028
8.0,0.021
9.0,0.016


### 2.6 Why is the ratio a whole number? What `minutes_played` in `player_performances` is

If the tables agreed, the ratio in 2.5 would be close to 1, not 2, 3, 4, ... Let us look at a few groups with ratio 3.0 together with the column `goals`.

In [26]:
look = mins[(mins["ratio"].round(1) == 3.0) & (mins["perf_rows"] == 1)].head(8)
look.merge(player_performances[CMP_KEY + ["goals", "minutes_played"]], on=CMP_KEY)[
    ["player_id", "season_start", "goals", "minutes_played", "app_minutes", "ratio"]]

,player_id,season_start,goals,minutes_played,app_minutes,ratio
0,215,2014,3.0,304.0,912.0,3.000000
1,691,2012,3.0,770.0,2311.0,3.001299
2,1791,2014,3.0,824.0,2472.0,3.000000
3,2865,2014,3.0,817.0,2451.0,3.000000
4,2923,2013,3.0,569.0,1707.0,3.000000
5,2998,2015,3.0,1050.0,3150.0,3.000000
6,3259,2012,3.0,779.0,2337.0,3.000000
7,3291,2013,3.0,950.0,2849.0,2.998947


In [27]:
pp = player_performances
kind = np.where(pp["goals"].isna(), "goals missing (goalkeeper)",
                np.where(pp["goals"] > 0, "goals > 0", "goals = 0"))

pd.crosstab(kind, pp["minutes_played"].notna(), normalize="index").round(3).rename(columns={False: "minutes_played missing", True: "minutes_played filled"})

minutes_played,minutes_played missing,minutes_played filled
row_0,,
goals = 0,1.0,0.0
goals > 0,0.0,1.0
goals missing (goalkeeper),0.0,1.0


### 2.7 Can the minutes in `player_performances` be rebuilt?

In 2.6, `minutes_played` in `player_performances` turned out not to be a plain number of minutes. For players with goals it holds minutes per goal, and for players with goals missing it holds real minutes. For players with 0 goals it is empty. Here we test whether the minutes can still be recovered. For players with goals we use `minutes_played × goals`, and for goals missing we take `minutes_played` as it is. We compare the result with `app_minutes` from `appearances`.

Only groups with one row in `player_performances` are used (one club per league and season), so nothing has to be summed over several clubs.

In [28]:
# 2.7 Can the minutes in player_performances be rebuilt from goals? ---------------
# Only groups with one row in player_performances (one club per league and season)
single = both[both["perf_rows"] == 1].merge(
    p_top5[CMP_KEY + ["goals", "minutes_played"]], on=CMP_KEY, how="left")

single["kind"] = np.where(single["goals"].isna(), "goals missing",
                 np.where(single["goals"] > 0, "goals > 0", "goals = 0"))

# goals > 0: minutes per goal x goals; goals missing: the minutes as they are
single["rebuilt"] = np.where(single["kind"] == "goals > 0", single["minutes_played"] * single["goals"],
                    np.where(single["kind"] == "goals missing", single["minutes_played"], np.nan))
single["rel_err"] = (single["app_minutes"] - single["rebuilt"]).abs() / single["app_minutes"]

g = single.groupby("kind")
res = pd.DataFrame({
    "groups": g.size(),
    "% of groups": (g.size() / len(single) * 100).round(1),
    "% of appearances minutes": (g["app_minutes"].sum() / single["app_minutes"].sum() * 100).round(1),
    "exact %": (g["rel_err"].apply(lambda s: (s.dropna() == 0).mean()) * 100).round(1),
    "within 1% ": (g["rel_err"].apply(lambda s: (s.dropna() <= 0.01).mean()) * 100).round(1),
})

# who are the players with goals missing?
gk = single[single["kind"] == "goals missing"].merge(players[["player_id", "position"]], on="player_id", how="left")
print("position of the groups with goals missing:", gk["position"].value_counts(dropna=False).to_dict())

# coverage: where can the minutes be rebuilt at all, by league and by season
single["rebuildable"] = single["kind"] != "goals = 0"
print("groups with one row in player_performances (one club per league and season):", len(single), "of", len(both))
print("minutes can be rebuilt for", single["rebuildable"].sum(), "groups =",
      round(single["rebuildable"].mean() * 100, 1), "% of groups and",
      round(single.loc[single["rebuildable"], "app_minutes"].sum() / single["app_minutes"].sum() * 100, 1),
      "% of appearances minutes")
print("share of groups by league (%):", (single.groupby("competition_id")["rebuildable"].mean() * 100).round(1).to_dict())
by_season = single.groupby("season_start")["rebuildable"].mean() * 100
print("share of groups by season (%): min", by_season.min().round(1), "| max", by_season.max().round(1))
res

position of the groups with goals missing: {'Goalkeeper': 2477}
groups with one row in player_performances (one club per league and season): 31364 of 32509
minutes can be rebuilt for 17908 groups = 57.1 % of groups and 73.5 % of appearances minutes
share of groups by league (%): {'ES1': 56.8, 'FR1': 55.1, 'GB1': 58.3, 'IT1': 56.4, 'L1': 59.1}
share of groups by season (%): min 55.4 | max 60.9


,groups,% of groups,% of appearances minutes,exact %,within 1%
kind,,,,,
goals = 0,13456,42.9,26.5,NaN,NaN
goals > 0,15431,49.2,64.2,54.1,99.4
goals missing,2477,7.9,9.3,98.8,99.1


### Findings

- **Season:** the season of a match is taken from `games.season`, not from the date (the July–June rule gives another season for 6,726 top-5 rows). The comparison covers 2012–2024, because `player_performances` is almost empty for 2025/26.
- **Completeness:** `player_performances` has fewer appearances per match in the early seasons (24.0 in 2012 against 27.6 in `appearances`). The gap narrows to under one appearance per match from 2018.
- **Groups (player, season, league):** 32,509 are in both tables. 6,357 are only in `player_performances`; 92.8% of them are squad members who never played, and 457 played but are missing from `appearances`. 2,144 are only in `appearances` (411 in 2012, 85 in 2024).
- **Appearances:** in groups present in both tables, the number of matches is equal in 99.8%, and only 5 groups differ by 3 or more.
- **Minutes:** the two columns do not measure the same thing. In `player_performances`, `minutes_played` is minutes per goal for players with goals, empty for players with 0 goals, and real minutes for goalkeepers. Rebuilt this way, it agrees with `appearances` within 1% in 99% of groups. But it can be rebuilt for only 57.1% of groups (73.5% of minutes), in 55–59% of groups in every league.
- **Consequence:** minutes and match dates come from `appearances`; `player_performances` serves only as a cross-check.

## Step 3: From when to when are the data complete?

**Question.** Is the recording of injuries and matches even over time, and for which period are all sources complete?

### 3.1 Injuries over time

We count injuries by the year of `from_date`: all injuries in `player_injuries`, the cleaned meniscal injuries (`m`), and those meniscal injuries whose two windows lie inside the `appearances` period. If recording were even, the share of meniscal injuries among all injuries would stay roughly constant.

In [29]:
# 3.1 Injuries per year: all, meniscal, and meniscal with both windows inside the appearances period
by_year = pd.DataFrame({
    "all injuries": player_injuries["from_date"].dt.year.value_counts().sort_index(),
    "meniscal (cleaned)": m["from_date"].dt.year.value_counts().sort_index(),
    "meniscal, windows inside appearances": m[m["in_range"]]["from_date"].dt.year.value_counts().sort_index(),
}).fillna(0).astype(int)
by_year.index = by_year.index.astype(int)
by_year["meniscal % of all"] = (100 * by_year["meniscal (cleaned)"] / by_year["all injuries"]).round(1)
by_year.loc[2008:2025]

,all injuries,meniscal (cleaned),"meniscal, windows inside appearances",meniscal % of all
from_date,,,,
2008,1942,31,0,1.6
2009,1879,70,0,3.7
2010,1923,50,0,2.6
2011,2722,58,0,2.1
2012,3103,82,0,2.6
2013,3867,60,36,1.6
2014,4123,76,76,1.8
2015,5405,100,100,1.9
2016,6822,101,101,1.5


Where does recording end? Injuries recorded in every month of 2025:

In [30]:
# Where does recording end? Injuries per month in 2025
recent = player_injuries[player_injuries["from_date"] >= "2025-01-01"]
months = pd.period_range("2025-01", recent["from_date"].max().to_period("M"), freq="M")
recent.groupby(recent["from_date"].dt.to_period("M")).size().reindex(months, fill_value=0).rename("injuries recorded").to_frame().T

,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
injuries recorded,1493,1599,1508,1370,997,240,761,1069,177,1,0,1


### 3.2 Completeness of `appearances` for top-5 matches

**Per match.** A match has two starting elevens, so **at least 22 entries** in `appearances`. For every season we count the top-5 matches in `games`, how many have any entries, how many have none, and how many have fewer than 22.

In [31]:
# 3.2 Per match: entries in appearances for every top-5 match
g5 = games[games["competition_id"].isin(TOP5)].copy()
g5["n_app"] = g5["game_id"].map(appearances.groupby("game_id").size()).fillna(0)

by_season = g5.groupby("season").agg(matches=("game_id", "size"),
                                     with_entries=("n_app", lambda s: (s > 0).sum()),
                                     no_entries=("n_app", lambda s: (s == 0).sum()),
                                     fewer_than_22=("n_app", lambda s: ((s > 0) & (s < 22)).sum()))
display(by_season)
print("top-5 matches in 2014/15 with fewer than 22 entries, by league:",
      g5[(g5["season"] == 2014) & (g5["n_app"] < 22)]["competition_id"].value_counts().to_dict())

,matches,with_entries,no_entries,fewer_than_22
season,,,,
2012,1826,1825,1,0
2013,1826,1826,0,0
2014,1826,1826,0,114
2015,1826,1826,0,0
2016,1826,1826,0,0
2017,1826,1826,0,0
2018,1826,1826,0,0
2019,1725,1725,0,0
2020,1826,1826,0,2


top-5 matches in 2014/15 with fewer than 22 entries, by league: {'ES1': 74, 'GB1': 38, 'IT1': 1, 'FR1': 1}


**Per club side of a match.** The criterion "fewer than 22 entries" misses matches in which only one side's squad list is missing. For every club in every top-5 match we count its entries; a side with **fewer than 11** entries is incomplete (a side with none is *absent*).

In [32]:
# 3.2 Per club side of a match
side = (appearances[appearances["competition_id"].isin(TOP5)]
        .groupby(["game_id", "player_club_id"]).size().rename("n").reset_index())
ev = pd.concat([g5[["game_id", "season", "competition_id", "date", "home_club_id"]].rename(columns={"home_club_id": "club"}),
                g5[["game_id", "season", "competition_id", "date", "away_club_id"]].rename(columns={"away_club_id": "club"})])
ev = ev.merge(side, left_on=["game_id", "club"], right_on=["game_id", "player_club_id"], how="left")
ev["n"] = ev["n"].fillna(0)
ev["short_side"] = ev["n"] < 11

club_names = pd.concat([games[["home_club_id", "home_club_name"]].set_axis(["id", "name"], axis=1),
                        games[["away_club_id", "away_club_name"]].set_axis(["id", "name"], axis=1)]).drop_duplicates("id").set_index("id")["name"]

sides = ev.groupby("season").agg(club_sides=("game_id", "size"), incomplete=("short_side", "sum"), absent=("n", lambda s: (s == 0).sum()))
display(sides[sides["incomplete"] > 0])

s14 = ev[(ev["season"] == 2014) & ev["short_side"]]
print("season 2014/15, clubs with the most incomplete sides:",
      {club_names.get(k, k): int(v) for k, v in s14["club"].value_counts().head(6).items()})
print("season 2014/15, incomplete sides of the three clubs with the most:", int(s14["club"].isin(s14["club"].value_counts().head(3).index).sum()), "of", len(s14))

,club_sides,incomplete,absent
season,,,
2012,3652,2,2
2014,3652,140,13
2018,3652,1,0
2020,3652,31,1
2021,3652,3,0
2025,3504,2,2


season 2014/15, clubs with the most incomplete sides: {'Burnley FC': 38, 'Atlético de Madrid': 38, 'Villarreal CF': 38, 'OGC Nice': 14, 'Cagliari Calcio': 12}
season 2014/15, incomplete sides of the three clubs with the most: 114 of 140


### 3.3 Players who played, according to `player_performances`, but are absent from `appearances`

Here we use the comparison table `c` from 2.3. A *gap* is a (player, season, league) group in which `player_performances` shows at least one appearance, but `appearances` has no row at all. For every season: how many such groups, how many matches they contain, and what share of all top-5 matches in `player_performances` that is.

In [33]:

# 3.3 Gaps: groups that played according to player_performances but have no row in appearances
pf_played = c[(c["_merge"] != "right_only") & (c["perf_games"] > 0)]
gaps = pf_played[pf_played["_merge"] == "left_only"]

gap_table = pd.DataFrame({
    "groups that played": pf_played.groupby("season_start").size(),
    "gap groups": gaps.groupby("season_start").size(),
    "matches in gap groups": gaps.groupby("season_start")["perf_games"].sum(),
}).fillna(0).astype(int)
gap_table["% of matches"] = (100 * gap_table["matches in gap groups"] / pf_played.groupby("season_start")["perf_games"].sum()).round(2)
gap_table

,groups that played,gap groups,matches in gap groups,% of matches
season_start,,,,
2012,2259,6,38,0.09
2013,2301,4,7,0.02
2014,2343,106,2071,4.51
2015,2459,2,29,0.06
2016,2461,0,0,0.00
2017,2477,1,1,0.00
2018,2476,35,78,0.16
2019,2584,5,93,0.19
2020,2697,20,555,1.04


The reverse direction: the share of **matches in `appearances`** that lie in groups absent from `player_performances`.

In [34]:
# 3.3 The reverse direction
ar = c[c["_merge"] != "left_only"]
miss_pp = ar[ar["_merge"] == "right_only"]
(100 * miss_pp.groupby("season_start")["app_games"].sum() / ar.groupby("season_start")["app_games"].sum()).round(2).rename("% of appearances matches absent from performances").to_frame().T

season_start,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
% of appearances matches absent from performances,13.04,11.93,9.05,7.34,5.97,4.14,3.6,2.1,1.85,1.94,1.81,2.31,2.35


### 3.4 The two gaps in detail: 2014/15 and 2021/22

In 3.3 the gap groups concentrate in two seasons: 2014/15 (106 groups) and 2021/22 (274 groups). We rebuild the gap groups with the club (`team_id`) and look at who and where they are.

In [35]:
# 3.4 Rebuild the gap groups with the club (team_id)
p5 = player_performances[player_performances["competition_id"].isin(TOP5) & (player_performances["nb_on_pitch"] > 0)]
miss = p5.merge(app_g[CMP_KEY], on=CMP_KEY, how="left", indicator="_in_app")
miss = miss[(miss["_in_app"] == "left_only") & miss["season_start"].between(FIRST_SEASON, LAST_SEASON)]
miss_g = miss.groupby(CMP_KEY).agg(matches=("nb_on_pitch", "sum"), team_id=("team_id", "first")).reset_index()

for S in (2014, 2021):
    s = miss_g[miss_g["season_start"] == S]
    print(f"=== season {S}/{str(S + 1)[2:]}: {len(s)} gap groups, {int(s['matches'].sum())} matches ===")
    print("by league:", s["competition_id"].value_counts().to_dict())
    print("clubs:", s["team_id"].nunique(), "| most frequent:", {club_names.get(k, k): int(v) for k, v in s["team_id"].value_counts().head(5).items()})
    print("matches per group, percentiles 10/50/90:", s["matches"].quantile([.1, .5, .9]).tolist(),
          "| groups with exactly 1 match:", int((s["matches"] == 1).sum()), "| with 10 or more:", int((s["matches"] >= 10).sum()))

=== season 2014/15: 106 gap groups, 2071 matches ===
by league: {'ES1': 53, 'IT1': 22, 'GB1': 19, 'FR1': 10, 'L1': 2}
clubs: 26 | most frequent: {'Atlético de Madrid': 21, 'Villarreal CF': 21, 'Burnley FC': 18, 'OGC Nice': 5, 'Cagliari Calcio': 5}
matches per group, percentiles 10/50/90: [2.0, 21.0, 34.5] | groups with exactly 1 match: 8 | with 10 or more: 78
=== season 2021/22: 274 gap groups, 1114 matches ===
by league: {'ES1': 88, 'IT1': 64, 'FR1': 64, 'L1': 29, 'GB1': 29}
clubs: 90 | most frequent: {'Venezia FC': 9, 'FC Girondins Bordeaux': 8, 'RCD Espanyol Barcelona': 8, 'AS Saint-Étienne': 8, 'Genoa CFC': 7}
matches per group, percentiles 10/50/90: [1.0, 2.0, 10.0] | groups with exactly 1 match: 119 | with 10 or more: 32


For season 2021/22 we also ask: are these players absent from `appearances` in the whole season, or only in one league? And are the match lists of that season incomplete (3.2)?

In [36]:

# 3.4 Season 2021/22: are these players absent from appearances in the whole season, or only in one league?
s21 = miss_g[miss_g["season_start"] == 2021]
any_app = appearances[appearances["season_start"] == 2021].groupby("player_id").size()
print("2021 gap players with ANY row in appearances in season 2021 (any competition):", int(s21["player_id"].map(any_app).notna().sum()), "of", len(s21))
for S in (2020, 2022):
    ids = set(appearances.loc[appearances["season_start"] == S, "player_id"])
    print(f"...with any row in appearances in season {S}:", int(s21["player_id"].isin(ids).sum()))
pos = s21.merge(players[["player_id", "position"]], on="player_id", how="left")["position"].value_counts(dropna=False)
print("positions of 2021 gap players:", pos.to_dict())

side21 = ev[ev["season"] == 2021]
print("2021/22 club sides with fewer than 11 entries:", int(side21["short_side"].sum()), "| absent sides:", int((side21["n"] == 0).sum()), "of", len(side21))

2021 gap players with ANY row in appearances in season 2021 (any competition): 0 of 274
...with any row in appearances in season 2020: 111
...with any row in appearances in season 2022: 93
positions of 2021 gap players: {'Attack': 116, 'Midfield': 83, 'Defender': 71, 'Goalkeeper': 4}
2021/22 club sides with fewer than 11 entries: 3 | absent sides: 0 of 3652


### 3.5 Who is in `player_performances` at all?

So far we asked who is missing from `appearances`. Here we ask what `player_performances` contains. If it contained only injured players, uninjured players could not be found in it. We count how many players with an injury record are in `player_performances`, how many of its players have any injury record, and, season by season, how many of the top-5 players of `appearances` are in `player_performances` and have an injury record.

In [37]:
# 3.5 Who is in player_performances at all, and who has an injury record?
inj_players = set(player_injuries["player_id"])
pp_players = set(player_performances["player_id"])
print("players with an injury record:", len(inj_players), "| of them in player_performances:", len(inj_players & pp_players),
      f"({len(inj_players & pp_players) / len(inj_players):.1%})")
print("players in player_performances:", len(pp_players), "| of them with any injury record:", len(inj_players & pp_players),
      f"({len(inj_players & pp_players) / len(pp_players):.1%})")

pairs = appearances.loc[appearances["competition_id"].isin(TOP5), ["season_start", "player_id"]].drop_duplicates()
pairs["in_performances"] = pairs["player_id"].isin(pp_players)
pairs["has_injury_record"] = pairs["player_id"].isin(inj_players)
pairs.groupby("season_start")[["in_performances", "has_injury_record"]].mean().mul(100).round(1).rename(
    columns={"in_performances": "% of top-5 players (appearances) present in performances", "has_injury_record": "% with any injury record"})

players with an injury record: 34561 | of them in player_performances: 34524 (99.9%)
players in player_performances: 88375 | of them with any injury record: 34524 (39.1%)


,% of top-5 players (appearances) present in performances,% with any injury record
season_start,,
2012,84.6,78.8
2013,86.2,81.7
2014,89.2,85.0
2015,91.4,87.2
2016,92.7,89.7
2017,94.6,91.0
2018,95.4,92.5
2019,97.1,93.1
2020,97.7,92.9


### Findings

- **Recording grows over time.** All injuries rise from 3,103 (2012) to 15,164 (2021) and 14,778 (2024), meniscal injuries from 82 to 148 and 154, so the meniscal share falls from 2.6% to 0.9–1.3% in 2018–2024. Yearly counts therefore measure recording, not frequency. Recording ends in September 2025.
- **Injuries usable in time.** Meniscal injuries with both windows inside the `appearances` period run from 2013 to 2024.
- **`appearances`, top 5.** Only two matches have no entries. The gaps are concentrated in 2014/15: 114 matches with fewer than 22 entries and 140 club sides with fewer than 11, 114 of them in three clubs (Burnley, Atlético de Madrid, Villarreal). Every other season has at most 31 such sides (2020).
- **Players who played but are absent from `appearances`:** 457 groups, mostly in 2014 (106; 4.51% of matches in `player_performances`) and 2021 (274; 2.07%). In 2014/15 they are players with many matches from the same three clubs. In 2021/22 they are players with few matches in 90 clubs, absent from `appearances` for the whole season, and the cause is unknown. In the other direction, the share of `appearances` matches absent from `player_performances` falls from 13.04% (2012) to about 2% from 2019.
- **Who is in `player_performances`.** Nearly all players with an injury record are in it (99.9%), but only 39.1% of its players have one.

## Step 4: How good are the meniscal injury records?

**Question.** Are the records of meniscal injuries consistent (dates, length of absence), complete (missing values), unique (duplicates), and do they overlap with other events that would distort the result?

### 4.1 Missing values and consistency of dates

We take all meniscal rows before cleaning and count the missing values in the date and absence columns. For the rows with both dates we check three things: is `end_date` ever before `from_date`, does `days_missed` equal `end_date − from_date + 1`, and how long are the longest absences.

In [38]:
# 4.1 Missing values and consistency of dates (all meniscal rows, before cleaning)
d = meniscus.copy()
print("rows (meniscal injuries):", len(d))
print("missing values:")
print(d[["from_date", "end_date", "days_missed", "games_missed"]].isna().sum().to_string())

ok = d.dropna(subset=["from_date", "end_date"])
calc_days = (ok["end_date"] - ok["from_date"]).dt.days + 1
print("\nend_date before from_date:", (ok["end_date"] < ok["from_date"]).sum())
print("days_missed different from end_date - from_date + 1:", (ok["days_missed"] != calc_days).sum())
print("absence longer than 365 days:", (ok["days_missed"] > 365).sum(), "| longer than 730:", (ok["days_missed"] > 730).sum(),
      "| longest:", int(ok["days_missed"].max()))

rows (meniscal injuries): 1863
missing values:
from_date        1
end_date        24
days_missed      1
games_missed     0

end_date before from_date: 0
days_missed different from end_date - from_date + 1: 0
absence longer than 365 days: 34 | longer than 730: 4 | longest: 838


### 4.2 Injuries without `end_date`

We saw that 24 meniscal injuries have no `end_date`. Where do they come from? Transfermarkt still gives `days_missed` for them, so we compute the date up to which it counts their absence (`from_date + days_missed`) and look at them by year of the start.

In [39]:
# 4.2 Injuries without end_date: up to which date does Transfermarkt count their absence?
open_inj = meniscus[meniscus["end_date"].isna() & meniscus["from_date"].notna()].copy()
open_inj["counted_until"] = open_inj["from_date"] + pd.to_timedelta(open_inj["days_missed"], unit="D")

print("injuries without end_date (with a start date):", len(open_inj))
print("date up to which absence is counted (from_date + days_missed):")
print(open_inj["counted_until"].dt.date.value_counts().to_string())
display(open_inj.groupby(open_inj["from_date"].dt.year).agg(injuries=("player_id", "size"),
                                                         absence_min=("days_missed", "min"), absence_max=("days_missed", "max")))

injuries without end_date (with a start date): 24
date up to which absence is counted (from_date + days_missed):
counted_until
2025-09-14    13
2025-09-13    11


,injuries,absence_min,absence_max
from_date,,,
2015,1,3718.0,3718.0
2020,1,1940.0,1940.0
2021,2,1361.0,1491.0
2022,1,1056.0,1056.0
2023,2,836.0,985.0
2024,2,321.0,599.0
2025,15,12.0,213.0


The date up to which absence is counted is the day on which the data were collected. Two more questions: how many injuries have an `end_date` after that day (a forecast end, not an observed one), and how many injuries without `end_date` could still have both windows inside the `appearances` period?

In [40]:
# 4.2 Do these injuries, or injuries with an end after the collection date, matter for our windows?
COLLECTED = open_inj["counted_until"].max()            # the day on which the data were collected
future_end = ok[ok["end_date"] > COLLECTED]
print("injuries with end_date AFTER the collection date (a forecast end):", len(future_end))
print("...of which have both windows inside appearances:", int((m["in_range"] & (m["end_date"] > COLLECTED)).sum()))

cand = open_inj[(open_inj["from_date"] >= app_min + pd.DateOffset(months=PRE_MONTHS))
                & (open_inj["from_date"] <= app_max - pd.DateOffset(months=POST_MONTHS))]
print("injuries without end_date that could still have both windows inside the data:", len(cand))

injuries with end_date AFTER the collection date (a forecast end): 12
...of which have both windows inside appearances: 0
injuries without end_date that could still have both windows inside the data: 8


### 4.3 Duplicates and near-duplicates


There are no full duplicates among the meniscal injuries (1.3). Here we look for the same player with the same start date but a different label or end date: that would be one injury recorded twice.

In [41]:
# 4.3 Same player, same start date
same_start = m[m.duplicated(["player_id", "from_date"], keep=False)].sort_values(["player_id", "from_date"])
print("rows with the same player and start date:", len(same_start))
same_start[["player_id", "injury_reason", "from_date", "end_date", "days_missed", "in_range"]]

rows with the same player and start date: 4


,player_id,injury_reason,from_date,end_date,days_missed,in_range
1592,7236,Meniscus tear,1998-12-14,1999-01-10,28.0,False
1593,7236,Meniscus irritation,1998-12-14,1999-01-10,28.0,False
1392,59084,Meniscus injury,2019-10-06,2019-11-19,45.0,True
1393,59084,Meniscus tear,2019-10-06,2019-11-18,44.0,True


### 4.4 Several meniscal injuries of one player

How many players have more than one meniscal injury, and how far apart are they? The gap is counted from the end of the previous meniscal injury to the start of the next one (negative = overlap). The column `n_men_before` stores how many earlier meniscal injuries of the same player are in the data.

In [42]:
# 4.4 Several meniscal injuries of one player
order = m.sort_values(["player_id", "from_date"])
m["n_men_before"] = order.groupby("player_id").cumcount().reindex(m.index)
order = order.assign(gap=order["from_date"] - order.groupby("player_id")["end_date"].shift())

per_player = m["player_id"].value_counts().value_counts().sort_index()
print("meniscal injuries per player (injuries: players):", per_player.to_dict())

gap_days = order["gap"].dt.days.dropna()
print("\ngap from the previous meniscal injury (days):")
print("  overlap (below 0):", (gap_days < 0).sum(), "| 0-30:", gap_days.between(0, 30).sum(),
      "| 31-365:", gap_days.between(31, 365).sum(), "| above 365:", (gap_days > 365).sum())

inr = m[m["in_range"]]
print("\nof", len(inr), "injuries with windows inside appearances:", (inr["n_men_before"] == 0).sum(),
      "are the player's first meniscal injury in the data,", (inr["n_men_before"] > 0).sum(), "are a later one; players:", inr["player_id"].nunique())

meniscal injuries per player (injuries: players): {1: 1453, 2: 157, 3: 18, 4: 3, 5: 1}

gap from the previous meniscal injury (days):
  overlap (below 0): 2 | 0-30: 25 | 31-365: 70 | above 365: 109

of 1277 injuries with windows inside appearances: 1135 are the player's first meniscal injury in the data, 142 are a later one; players: 1164


### 4.5 Other injuries nearby: cruciate ligament and the rest

Other injuries can distort what we see in the windows. We add three flags (columns in `m`) and do not remove any row:
- `acl_30d`: a cruciate ligament injury that starts from 30 days before the meniscal injury to 30 days after its end,
- `other_during`: another (non-meniscal) injury that starts during the meniscal absence,
- `other_90d_after`: another injury that starts within 90 days after the return.

In [43]:
# 4.5 Other injuries near the meniscal injury: three flags
acl_labels = player_injuries.loc[player_injuries["injury_reason"].str.contains("cruciate", case=False, na=False), "injury_reason"].unique()
other = player_injuries[~player_injuries["injury_reason"].isin(MENISCUS_LABELS)].dropna(subset=["from_date"])

x = m[["inj_id", "player_id", "from_date", "end_date"]].merge(
    other[["player_id", "injury_reason", "from_date"]].rename(columns={"injury_reason": "o_reason", "from_date": "o_from"}),
    on="player_id")

acl_near = (x["o_reason"].isin(acl_labels) & (x["o_from"] >= x["from_date"] - pd.Timedelta(days=30))
            & (x["o_from"] <= x["end_date"] + pd.Timedelta(days=30)))
during  = (x["o_from"] >= x["from_date"]) & (x["o_from"] <= x["end_date"])
after90 = (x["o_from"] > x["end_date"]) & (x["o_from"] <= x["end_date"] + pd.Timedelta(days=90))

m["acl_30d"]         = m["inj_id"].isin(x.loc[acl_near, "inj_id"])
m["other_during"]    = m["inj_id"].isin(x.loc[during, "inj_id"])
m["other_90d_after"] = m["inj_id"].isin(x.loc[after90, "inj_id"])

print("ACL labels in the data:", list(acl_labels))
pd.DataFrame({"all meniscal injuries (m)": m[["acl_30d", "other_during", "other_90d_after"]].sum(),
              "windows inside appearances": m.loc[m["in_range"], ["acl_30d", "other_during", "other_90d_after"]].sum()})

ACL labels in the data: ['Cruciate ligament tear', 'Cruciate ligament injury', 'Cruciate ligament surgery', 'Cruciate ligament strain', 'Partial damage to the cruciate ligament']


,all meniscal injuries (m),windows inside appearances
acl_30d,42,29
other_during,116,77
other_90d_after,301,207


### Findings

- **Records are consistent:** `days_missed` equals `end_date − from_date + 1` in every row with both dates, and `end_date` is never before `from_date`. 24 rows lack `end_date` and one lacks `from_date`; the longest absence is 838 days.
- **Injuries without `end_date`:** Transfermarkt counts all 24 up to the collection date (2025-09-13 or 14). 15 start in 2025; 9 start in 2015–2024 with absences of 321–3,718 days, so they are probably records with no end entered. Only 8 could have both windows inside `appearances`. 12 injuries have an `end_date` after the collection date (a forecast), and none of them has both windows inside.
- **Duplicates and repeats:** two pairs of rows share player and start date, and one pair (player 59084) lies inside the `appearances` period. 179 of 1,632 players have more than one meniscal injury; of the 1,277 injuries with windows inside, 142 are not the player's first.
- **Other injuries nearby** (flags; all 1,838 / the 1,277 inside): cruciate ligament within 30 days 42 / 29, another injury during the absence 116 / 77, another within 90 days after the return 301 / 207.